# 02 · Ontology mapping and label granularity (Finding 1)
How specific is each dataset's labelling? We use the number of Cell Ontology ancestors as a depth score (more ancestors = more specific).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from celltype_rel.ontology import label, depth, parser

obs = pd.read_parquet("../data/processed/blood_obs.parquet")
CHOSEN = []  # same list as notebook 01
obs = obs[obs.dataset_id.isin(CHOSEN)] if CHOSEN else obs
terms = pd.DataFrame({"term": obs.cell_type_ontology_term_id.unique()})
terms["label"] = terms.term.map(label)
terms["depth"] = terms.term.map(depth)
terms.sort_values("depth")

In [ ]:
# per-dataset summary
obs["depth"] = obs.cell_type_ontology_term_id.map(terms.set_index("term").depth)
summary = obs.groupby("dataset_id").agg(n_labels=("cell_type", "nunique"),
                                        mean_depth=("depth", "mean"))
summary

In [ ]:
# dataset x cell type: share of each dataset's cells per label
mat = pd.crosstab(obs.cell_type, obs.dataset_id, normalize="columns")
plt.figure(figsize=(8, 14))
sns.heatmap(mat > 0, cbar=False)
plt.title("Which labels does each dataset use?")

**Next:** group labels by lineage (T, B, NK, monocyte, …) with `parser().get_term_ancestors` and draw the sub-tree for T cells, marking which dataset uses which node.